In [6]:
import torch 
from torch.utils.data import TensorDataset, DataLoader

In [10]:
def make_memory_data(n, length=30):
    """Random digit sequences; label = 1 if the FIRST digit is greater than 4."""
    x = torch.randint(0, 10, (n, length))                      # TODO: tensor (n, length) of random integers 0-9
    y = (x[:, 0] > 4).long()                       # TODO: tensor (n,) of 0/1 depending on the first column of x
    return x, y

In [11]:
x, y = make_memory_data(5, 8)
print(x)
print(y)

tensor([[4, 7, 7, 2, 3, 9, 2, 2],
        [1, 4, 3, 1, 5, 7, 0, 5],
        [7, 1, 5, 8, 6, 0, 1, 0],
        [5, 8, 7, 1, 2, 4, 2, 7],
        [0, 9, 0, 1, 4, 5, 1, 0]])
tensor([0, 0, 1, 1, 0])


In [26]:
x_train, y_train = make_memory_data(50000)
x_val, y_val = make_memory_data(1000)

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=64, shuffle=False)

In [27]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = "cuda" if torch.cuda.is_available() else "cpu"


class SentimentNet(nn.Module):
    """Embedding -> LSTM or RNN -> last hidden state -> Linear."""

    def __init__(self, vocab_size, emb_dim, hidden_dim, model_type="lstm"):
        """Create the layers (runs once, when the model is built)."""
        super().__init__()
        # Token index -> vector of emb_dim numbers (index 0 is kept as zeros)
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)

        if model_type == "lstm":
            self.recurrent = nn.LSTM(emb_dim, hidden_dim, batch_first=True)
        else:
            self.recurrent = nn.RNN(emb_dim, hidden_dim, batch_first=True)

        # Turns the final summary into 2 scores
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        """Use the layers on every batch. x: (batch, length)."""
        emb = self.embedding(x)                # (batch, length, emb_dim)
        out, state = self.recurrent(emb)       # LSTM: (h_n, c_n); RNN: h_n
        h_n = state[0] if isinstance(state, tuple) else state
        return self.fc(h_n[-1])                # (batch, 2)


def build_model(vocab_size, emb_dim, hidden_dim, model_type="lstm"):
    """Create the sentiment/memory model."""
    return SentimentNet(vocab_size, emb_dim, hidden_dim, model_type)


def evaluate(model, loader, loss_fn, device):
    """Return (average loss, accuracy) of the model on a loader."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += loss_fn(logits, yb).item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)
    return total_loss / total, correct / total


def fit(model, train_loader, val_loader, epochs, lr, device, clip=1.0):
    """Train the model and return the history of losses and accuracies."""
    model.to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = loss_fn(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), clip)   # avoid exploding gradients
            optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)

        train_loss, train_acc = total_loss / total, correct / total
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        for key, value in zip(history, (train_loss, val_loss, train_acc, val_acc)):
            history[key].append(value)
        print(f"Epoch {epoch}/{epochs} | train loss {train_loss:.3f} acc {train_acc:.3f} "
              f"| val loss {val_loss:.3f} acc {val_acc:.3f}")
    return history

In [29]:
for model_type in ["rnn", "lstm"]:
    torch.manual_seed(42)
    net = build_model(vocab_size=10, emb_dim=8, hidden_dim=32, model_type=model_type)
    history = fit(net, train_loader, val_loader, 10, 5e-3, device, clip=1.0)
    print(model_type, "best val acc:", max(history["val_acc"]))

Epoch 1/10 | train loss 0.695 acc 0.500 | val loss 0.695 acc 0.479
Epoch 2/10 | train loss 0.694 acc 0.496 | val loss 0.695 acc 0.462
Epoch 3/10 | train loss 0.694 acc 0.502 | val loss 0.693 acc 0.506
Epoch 4/10 | train loss 0.181 acc 0.910 | val loss 0.137 acc 0.952
Epoch 5/10 | train loss 0.134 acc 0.954 | val loss 0.143 acc 0.948
Epoch 6/10 | train loss 0.153 acc 0.943 | val loss 0.306 acc 0.882
Epoch 7/10 | train loss 0.182 acc 0.944 | val loss 0.177 acc 0.944
Epoch 8/10 | train loss 0.185 acc 0.941 | val loss 0.204 acc 0.934
Epoch 9/10 | train loss 0.209 acc 0.927 | val loss 0.215 acc 0.926
Epoch 10/10 | train loss 0.227 acc 0.920 | val loss 0.281 acc 0.897
rnn best val acc: 0.952
Epoch 1/10 | train loss 0.694 acc 0.503 | val loss 0.693 acc 0.503
Epoch 2/10 | train loss 0.693 acc 0.498 | val loss 0.693 acc 0.486
Epoch 3/10 | train loss 0.693 acc 0.499 | val loss 0.694 acc 0.486
Epoch 4/10 | train loss 0.693 acc 0.500 | val loss 0.693 acc 0.504
Epoch 5/10 | train loss 0.693 acc 0.4

In [ ]:
import torch
import torch.nn as nn

rnn = nn.RNN(input_size=8, hidden_size=32, batch_first=True)
x = torch.randn(1, 30, 8)      # one sequence: 30 steps, 8 numbers each
out, h_n = rnn(x)
print(out.shape)    # memory after every step
print(h_n.shape)    # memory after the last step

In [32]:
torch.manual_seed(42)
net = build_model(vocab_size=10, emb_dim=8, hidden_dim=32, model_type="lstm")

# LSTM bias layout is [input | forget | cell | output] gates, 32 values each,
# so the forget gate is the slice [32:64]
with torch.no_grad():
    net.recurrent.bias_ih_l0[32:64].fill_(2.0)   # forget gate starts "open" (sigmoid(2) ~ 0.88)
    net.recurrent.bias_hh_l0[32:64].fill_(0.0)

history = fit(net, train_loader, val_loader, 15, 5e-3, device, clip=1.0)

Epoch 1/15 | train loss 0.230 acc 0.883 | val loss 0.000 acc 1.000
Epoch 2/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 3/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 4/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 5/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 6/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 7/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 8/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 9/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 10/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 11/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 12/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 13/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 14/15 | train loss 0.000 acc 1.000 | val loss 0.000 acc 1.000
Epoch 15/15 | train loss 0.000 acc 1.000 | val loss 0.000

In [31]:
print(0.5 ** 29)

1.862645149230957e-09


In [34]:
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers


def make_memory_data(n, length=30, seed=0):
    """Random digits 0-9; label is 1 if the FIRST digit is > 4."""
    rng = np.random.default_rng(seed)
    x = rng.integers(0, 10, size=(n, length))   # (n, length)
    y = (x[:, 0] > 4).astype("int64")           # (n,)
    return x, y


def build_model(model_type, vocab_size=10, emb_dim=8, hidden_dim=32):
    """Embedding -> RNN / LSTM / CNN -> Dense(2)."""
    model = keras.Sequential()
    model.add(layers.Embedding(vocab_size, emb_dim))   # (n, length, emb_dim)

    if model_type == "rnn":
        model.add(layers.SimpleRNN(hidden_dim))        # last hidden state
    elif model_type == "lstm":
        # unit_forget_bias=True is the Keras default (the trick we did by hand)
        model.add(layers.LSTM(hidden_dim))
    elif model_type == "cnn":
        # Conv1D already expects (batch, length, channels), no permute needed
        model.add(layers.Conv1D(hidden_dim, kernel_size=3, activation="relu"))
        model.add(layers.GlobalMaxPooling1D())
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    model.add(layers.Dense(2, activation="softmax"))
    model.compile(
        optimizer=keras.optimizers.Adam(5e-3, clipnorm=1.0),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def run_experiment(model_type, epochs=15):
    """Train one model on the toy memory task and return best val accuracy."""
    keras.utils.set_random_seed(42)
    x_train, y_train = make_memory_data(50000, length=10, seed=1)
    x_val, y_val = make_memory_data(5000, length=10, seed=2)

    model = build_model(model_type)
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs, batch_size=64, verbose=2,
    )
    best = max(history.history["val_accuracy"])
    print(f"{model_type}: best val acc {best:.3f}")
    return best


if __name__ == "__main__":
    for kind in ["rnn", "lstm", "cnn"]:
        run_experiment(kind)

Epoch 1/15
782/782 - 2s - 2ms/step - accuracy: 0.9970 - loss: 0.0070 - val_accuracy: 1.0000 - val_loss: 6.0511e-06
Epoch 2/15
782/782 - 1s - 1ms/step - accuracy: 1.0000 - loss: 3.6535e-06 - val_accuracy: 1.0000 - val_loss: 2.2013e-06
Epoch 3/15
782/782 - 1s - 1ms/step - accuracy: 1.0000 - loss: 1.5634e-06 - val_accuracy: 1.0000 - val_loss: 1.0977e-06
Epoch 4/15
782/782 - 1s - 1ms/step - accuracy: 1.0000 - loss: 8.3251e-07 - val_accuracy: 1.0000 - val_loss: 6.2370e-07
Epoch 5/15
782/782 - 1s - 1ms/step - accuracy: 1.0000 - loss: 4.8969e-07 - val_accuracy: 1.0000 - val_loss: 3.8111e-07
Epoch 6/15
782/782 - 1s - 1ms/step - accuracy: 1.0000 - loss: 3.0520e-07 - val_accuracy: 1.0000 - val_loss: 2.4550e-07
Epoch 7/15
782/782 - 1s - 1ms/step - accuracy: 1.0000 - loss: 1.9198e-07 - val_accuracy: 1.0000 - val_loss: 1.4615e-07
Epoch 8/15
782/782 - 1s - 1ms/step - accuracy: 1.0000 - loss: 1.2919e-07 - val_accuracy: 1.0000 - val_loss: 1.2126e-07
Epoch 9/15
782/782 - 1s - 1ms/step - accuracy: 1.000

In [35]:
for steps in [1, 5, 10, 20, 30]:
    print(steps, 0.5 ** steps)

1 0.5
5 0.03125
10 0.0009765625
20 9.5367431640625e-07
30 9.313225746154785e-10
